In [1]:
"""
Titanic Data Analysis and JSON Export
Author: [Your Name]
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


In [2]:
# Load the Titanic dataset into a DataFrame
df = pd.read_csv(CSV_FILE)

print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [3]:
# Select numeric columns only
numeric_columns = df.select_dtypes(include=[np.number]).columns

# Calculate statistics (.mean, median, std)
summary_stats = df[numeric_columns].agg(['mean', 'median', 'std'])
print(summary_stats)

        PassengerId  Survived    Pclass        Age     SibSp     Parch  \
mean     446.000000  0.383838  2.308642  29.699118  0.523008  0.381594   
median   446.000000  0.000000  3.000000  28.000000  0.000000  0.000000   
std      257.353842  0.486592  0.836071  14.526497  1.102743  0.806057   

             Fare  
mean    32.204208  
median  14.454200  
std     49.693429  


In [4]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)

missing_data = {}

for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    missing_data[col] = {'count': missing_count, 'percent': missing_percent}
    print(f"{col}: {missing_count} missing ({missing_percent:.2f}%)")


MISSING VALUES ANALYSIS
PassengerId: 0 missing (0.00%)
Survived: 0 missing (0.00%)
Pclass: 0 missing (0.00%)
Name: 0 missing (0.00%)
Sex: 0 missing (0.00%)
Age: 177 missing (19.87%)
SibSp: 0 missing (0.00%)
Parch: 0 missing (0.00%)
Ticket: 0 missing (0.00%)
Fare: 0 missing (0.00%)
Cabin: 687 missing (77.10%)
Embarked: 2 missing (0.22%)


In [ ]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()

# Feature 1: Family Size
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))

# Feature 2: Is Alone
df_features['IsAlone'] = df_features['FamilySize'] == 1
print(df_features[['FamilySize', 'IsAlone']].head(10))

# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'

df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))

# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)

print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)

print("\nIs Alone by Survival:")
print(df_features.groupby('Survived')['IsAlone'].mean())

print("\nAge Group by Survival:")
print(df_features.groupby(['Survived', 'AgeGroup']).size().unstack(fill_value=0))

print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)

survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]

print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2    False
1           2    False
2           1     True
3           2    False
4           1     True
5           1     True
6           1     True
7           5    False
8           3    False
9           2    False
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0 

In [6]:
# Step 3: Create Classes for JSON Export

class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass,
                 fare, embarked=None, family_size=None, is_alone=None, title=None):
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = bool(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None

    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title,
        }

class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []
        self._create_passengers()

    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        for idx, row in self.dataframe.iterrows():
            passenger_id = row.get('PassengerId', idx)
            name = row.get('Name', None)
            age = row.get('Age', None)
            sex = row.get('Sex', None)
            survived = row.get('Survived', None)
            pclass = row.get('Pclass', None)
            fare = row.get('Fare', None)
            embarked = row.get('Embarked', None)
            family_size = row.get('FamilySize', None)
            is_alone = row.get('IsAlone', None)
            title = row.get('Title', None)
            if title is None and pd.notna(name):
                title = str(name).split(',')[1].split('.')[0].strip() if ',' in str(name) else None
            self.passengers.append(Passenger(passenger_id, name, age, sex, survived, pclass, fare, embarked, family_size, is_alone, title))

    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': float(self.dataframe['Survived'].mean()) if 'Survived' in self.dataframe.columns else None,
            },
            'passengers': [p.to_dict() for p in self.passengers]
        }

        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)

        print(f"Data exported to {filename}")
        return data

    def get_summary_stats(self):
        """Get summary statistics."""
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        did_not_survive = sum(1 for p in self.passengers if p.survived == 0)
        average_age = sum(p.age for p in self.passengers if p.age is not None) / sum(1 for p in self.passengers if p.age is not None)
        average_fare = sum(p.fare for p in self.passengers if p.fare is not None) / sum(1 for p in self.passengers if p.fare is not None)

        return {
            'total_passengers': len(self.passengers),
            'survived': survived_count,
            'did_not_survive': did_not_survive,
            'average_age': average_age,
            'average_fare': average_fare,
        }

if 'df_features' in locals() and not df_features.empty:
    df_engineered = df_features
    dataset = TitanicDataset(df_engineered)
    print('Dataset created successfully!')
    print(dataset.get_summary_stats())
    dataset.to_json(JSON_FILE)
# Additional validation: Load and inspect JSON
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)
 
# Print summary of JSON data and verify content

Dataset created successfully!
{'total_passengers': 891, 'survived': 342, 'did_not_survive': 549, 'average_age': 29.69911764705882, 'average_fare': 32.204207968574636}
Data exported to data/titanic_data.json


#Answers
- Features Created
    - The data was grouped in the following ways, focusing on rate of survival by: whether someone was alone on the Titanic vs. with at least one other person, size of the family onboard, age vs. survival.
- How have features differentiated between survivors and non-survivors
    - Family Size by Survival:
        - The mean is larger for survivors than for people who died and the standard deviation is much tighter. This seems to imply that larger family sizes had an advantage in overall survival on the Titanic. Though the difference in family size is not that significant.
    - Is Alone by Survival
        - This compares whether you were alone vs. if you had at least one other person on board as family. In this case the difference was much more significant where people who were alone were 68% of deaths vs. 48% for people with someone or family members.
    - Age Group by Survival:
        - This tracked survival across different age groups with some interesting findings. Adults were about 50% more likely to die, children roughly equal chance, Seniors were nearly twice as likely to die, and young adults were 84% more likely to die. This gives children the best chance and supports other data pointing to having someone with you being better for survivability. One can assume that all children were not alone but that can’t be said for the other age groups.
- What have you learned about using classes for data processing
    - If you have different sets of data or different methods to transform a given dataset, grouping those together in separate classes keeps your data manageable.
    - If you have multiple data sets but want to apply the same transformations many times, you can use classes for reusable code. It also scales better to more complex architectures.
